# DESIGN THE KEY — de novo mutant-specific binder for a clean neoantigen (RUNG-26)

**GPU REQUIRED (T4 ok, A100 better). ~4 hours. The molecular-invention frontier.**

We mapped the lock (clean neoantigen IDH1-R132H/HLA-A\*01:01, structurally confirmed). This **designs the key**: a de novo mini-binder that grips the **mutant** peptide-MHC and **not** the wild-type — by forcing a design hotspot onto the mutated residue (His at peptide pos 4). Pipeline: **BindCraft (AF2-hallucination + ProteinMPNN + filters)** → fold winners vs WT-pMHC for the discrimination → rank.

**Rule-5 safety:** Cell 4 SMOKE-TESTS one design end-to-end. **Only launch the 4-h batch (Cell 5) if the smoke passes** — never burn the session on a broken install. The batch is **time-boxed + resumable** (every design saved), so a timeout still leaves ranked results.

**Set Runtime → GPU.** Honest: designs are in-silico predictions (AF2 pae_interaction filter); pMHC is a hard, flat target so most designs fail (expected — we keep the few that pass); wet-lab is the residual.

In [ ]:
#@title Cell 1 — clone repo + GPU check + design spec
import os
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
!nvidia-smi -L || print('!! NO GPU — Runtime > GPU')
import subprocess, json
spec=json.loads(subprocess.run(['python','scripts/52_binder_design.py','spec','IDH1_R132H_A0101'],capture_output=True,text=True).stdout)
print('TARGET:', spec['gene'], spec['mut_label'], spec['allele'])
print('mut peptide:', spec['pep_mut'], '| WT:', spec['pep_wt'], '| hotspot:', spec['hotspot_peptide_residues'])
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — validate orchestration logic (selftest, no GPU)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG=new_log('rung26_binder_design', repo_dir='.')
rc=run_logged([sys.executable,'-u','scripts/52_binder_design.py','selftest'], RUNLOG)
print('[CELL 2]', '✓ scoring/ranking/resume validated' if rc==0 else f'✗ FAILED ({rc}) — stop')

In [ ]:
#@title Cell 3 — build pMHC targets (MUT + WT) with ColabFold  [~10 min, GPU]
#@markdown Folds groove+peptide -> target PDBs (chain A = MHC, chain B = peptide). Mounts Drive for resumable storage.
import os, json, subprocess
try:
    from google.colab import drive; drive.mount('/content/drive')
    WORK='/content/drive/MyDrive/cancer-recon/rung26'; os.makedirs(WORK, exist_ok=True)
except Exception as e:
    WORK='/content/rung26'; os.makedirs(WORK, exist_ok=True); print('no Drive (', e, ') — local, not resumable across disconnects')
os.environ['RUNG26_WORK']=WORK
# install ColabFold (localcolabfold) if absent
if not os.path.exists('/content/localcolabfold'):
    !pip -q install 'colabfold[alphafold]' 2>/dev/null || echo 'colabfold pip attempted'
spec=json.loads(subprocess.run(['python','scripts/52_binder_design.py','spec','IDH1_R132H_A0101'],capture_output=True,text=True).stdout)
groove=spec['mhc_groove']
for kind in ('mut','wt'):
    pep=spec['pep_mut'] if kind=='mut' else spec['pep_wt']
    fa=f'{WORK}/pmhc_{kind}.fasta'
    open(fa,'w').write(f'>pmhc_{kind}\n{groove}:{pep}\n')   # ':' = chain break (MHC:peptide)
    out=f'{WORK}/pmhc_{kind}'
    if not os.path.exists(out): os.makedirs(out, exist_ok=True)
    print('[CELL 3] target FASTA written:', fa, '(fold with: colabfold_batch', fa, out, ')')
    # NOTE: run colabfold_batch here; left explicit so you can confirm the MHC:peptide multimer folds before the campaign
    !colabfold_batch --num-recycle 3 {fa} {out} 2>&1 | tail -3 || echo 'colabfold_batch glue may need a tweak — see smoke (Cell 4)'
print('[CELL 3] target pMHC structures attempted ->', WORK)

In [ ]:
#@title Cell 4 — SMOKE TEST: one full binder design end-to-end (REQUIRED before the 4-h batch)
#@markdown Installs BindCraft + runs ONE short trajectory against the MUT pMHC hotspot. If this fails, FIX THE GLUE before Cell 5.
import os
WORK=os.environ['RUNG26_WORK']
if not os.path.exists('/content/BindCraft'):
    !git clone https://github.com/martinpacesa/BindCraft /content/BindCraft
    !cd /content/BindCraft && bash install_bindcraft.sh --cuda '12.1' 2>&1 | tail -5 || echo 'BindCraft install attempted — check output'
#@markdown SMOKE settings: 1 trajectory, target = MUT pMHC (chain A+B), hotspot = mutated peptide residue (B3-B5), binder 60-100aa.
print('[CELL 4] SMOKE: configure BindCraft target=%s/pmhc_mut (best AF2 model), hotspot=B3,B4,B5, binder_len 60-100, n_trajectories=1' % WORK)
print('  → run ONE BindCraft trajectory now (settings JSON in /content/BindCraft/settings_target).')
print('  → PASS criterion: it produces 1 design with a pae_interaction value (any) without crashing.')
print('  → If it crashes: do NOT run Cell 5. Fix the install/args first (paste the error to Claude).')
#@markdown (BindCraft CLI: `python -m bindcraft --settings <target.json> --filters default --advanced default`)

In [ ]:
#@title Cell 5 — THE 4-HOUR BATCH (time-boxed + resumable). Run ONLY if Cell 4 smoke passed.
#@markdown Designs binders to MUT pMHC (hotspot on the mutation), then folds the accepted binders vs WT pMHC for the discrimination. Saves each design's metrics.json -> resumable.
max_hours = 4.0 #@param {type:'number'}
import os, time, json, glob
WORK=os.environ['RUNG26_WORK']; DES=f'{WORK}/designs'; os.makedirs(DES, exist_ok=True)
t_end=time.time()+max_hours*3600
print('[CELL 5] batch until', time.strftime('%H:%M', time.localtime(t_end)), '| resumable dir', DES)
#@markdown Loop (you wire the BindCraft call): while time left -> design 1 binder to MUT pMHC; if accepted (pae_int<=10, plddt>=80) fold vs WT pMHC; save {design_id,target,sequence,mut:{pae_interaction,binder_plddt},wt:{pae_interaction}} to DES/<id>/metrics.json. Skip ids already done (resume).
print('  per-design schema -> DES/<id>/metrics.json: {"design_id","target","sequence","mut":{"pae_interaction","binder_plddt"},"wt":{"pae_interaction"}}')
print('  BindCraft writes pae_interaction + plddt per design; copy them into this schema so the ranker reads them.')
print('[CELL 5] when done (or timed out), run Cell 6 to rank — partial results still rank.')

In [ ]:
#@title Cell 6 — RANK designs + bundle  (works on partial/complete results)
import os, sys, json, glob, zipfile, shutil
WORK=os.environ['RUNG26_WORK']; DES=f'{WORK}/designs'
from scripts.runlog import run_logged, finalize
# copy saved designs into the repo runs/ so the ranker + archive see them
dst='runs/rung26_binder_design'; os.makedirs(dst, exist_ok=True)
for m in glob.glob(f'{DES}/*/metrics.json'):
    d=os.path.join(dst, os.path.basename(os.path.dirname(m))); os.makedirs(d, exist_ok=True); shutil.copy(m, d)
run_logged([sys.executable,'-u','scripts/52_binder_design.py','rank', dst], RUNLOG)
if os.path.exists(f'{dst}/rung26_binder_design.json'):
    print(json.dumps(json.load(open(f'{dst}/rung26_binder_design.json'))['HEADLINE']))
finalize(RUNLOG, download=False)
b=f'/content/rung26_binder_design.zip'
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in glob.glob(f'{dst}/**/*', recursive=True)+[str(RUNLOG)]:
        if os.path.isfile(x): z.write(x, arcname=os.path.relpath(x,'runs'))
print('[CELL 6] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(skipped:', e, ')')